표에 적용하기

In [42]:
# 1. 템플릿 불러오기
# 2. 템플릿 xml 로 변환
# 3. 템플릿 json으로 변환
# 4. 템플릿 랜더링 json으로 변환
# 5.1 html로 렌더링 해보기
# 5.2 랜더링 json 변경해보기
# 6. 랜더링 json -> 한글로 압축하기


In [43]:
!pip -q install lxml xmltodict 


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# STEP1, STEP2, STEP3 함수 불러오기
from pathlib import Path
from zipfile import ZipFile

def extract_hwpx(hwpx_path: str | Path, output_dir: str | Path) -> Path:
    """한글 파일 압축 푸는 함수"""
    hwpx_path = Path(hwpx_path)
    output_dir = Path(output_dir)
    
    if not hwpx_path.exists():
        raise FileNotFoundError(f"hwpx file not found: {hwpx_path}")
    
    with ZipFile(hwpx_path, "r") as zip_file:
        zip_file.extractall(output_dir)
    return output_dir

def hwpx_xml_to_json(xml_dir, save_path):
    """ection0.xml, header.xml, settings.xml-> *.json"""
    from pathlib import Path
    import xml.etree.ElementTree as ET
    import json

    xml_dir = Path(xml_dir)

    targets = {
        "section": xml_dir / "Contents" / "section0.xml",
        "header": xml_dir / "Contents" / "header.xml",
        "settings": xml_dir / "settings.xml",
    }

    def strip_ns(tag):
        return tag.split("}", 1)[-1] if "}" in tag else tag

    def xml_to_dict(elem):
        return {
            "tag": strip_ns(elem.tag),
            "attrs": elem.attrib,
            "text": (elem.text or "").strip(),
            "children": [xml_to_dict(child) for child in elem]
        }

    hwpx_jsonb = {}

    for name, path in targets.items():
        tree = ET.parse(path)
        root = tree.getroot()

        hwpx_jsonb[name] = {
            "source_path": str(path),
            "root_tag": strip_ns(root.tag),
            "data": xml_to_dict(root)
        }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(hwpx_jsonb, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {save_path}")

    return hwpx_jsonb

def json_to_hwpx(json_data, template_dir, output_hwpx):
    """json을 xml로 변환 후, 한글템플릿과 치환하여 한글 파일로 변경하는 함수"""
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    import xml.etree.ElementTree as ET

    def json_to_xml_elem(data):
        elem = ET.Element(data["tag"], data.get("attrs", {}))

        text = data.get("text", "")
        if text:
            elem.text = text

        for child in data.get("children", []):
            elem.append(json_to_xml_elem(child))

        return elem

    template_dir = Path(template_dir)

    targets = {
        "section": template_dir / "Contents" / "section0.xml",
        "header": template_dir / "Contents" / "header.xml",
        "settings": template_dir / "settings.xml",
    }

    for key, xml_path in targets.items():
        root_data = json_data[key]["data"]
        root_elem = json_to_xml_elem(root_data)

        tree = ET.ElementTree(root_elem)
        ET.indent(tree, space="  ", level=0)
        tree.write(
            xml_path,
            encoding="utf-8",
            xml_declaration=True
        )

        print("XML 치환 완료:", xml_path)

    with ZipFile(output_hwpx, "w", ZIP_DEFLATED) as z:
        for file in template_dir.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(template_dir))

    print("HWPX 생성 완료:", output_hwpx)

    return output_hwpx

def xml_to_hwpx(xml_dir, template_dir, output_hwpx):
    """템플릿 HWPX 구조를 유지하고 XML만 치환하여 HWPX 생성"""

    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from shutil import copytree, copy2, rmtree
    import tempfile

    xml_dir = Path(xml_dir)
    template_dir = Path(template_dir)

    with tempfile.TemporaryDirectory() as temp_dir:

        work_dir = Path(temp_dir) / "hwpx"

        # 1. 템플릿 전체 복사
        copytree(template_dir, work_dir)

        # 2. XML 치환
        replace_files = [
            "Contents/section0.xml",
            "Contents/header.xml",
            "settings.xml",
        ]

        for rel_path in replace_files:

            src = xml_dir / rel_path
            dst = work_dir / rel_path

            if src.exists():
                copy2(src, dst)
                print("XML 치환:", rel_path)

        # 3. HWPX 생성
        with ZipFile(output_hwpx, "w", ZIP_DEFLATED) as z:

            for file in work_dir.rglob("*"):
                if file.is_file():
                    z.write(
                        file,
                        file.relative_to(work_dir)
                    )

    print("HWPX 생성 완료:", output_hwpx)

    return output_hwpx

### STEP2 함수
def make_render_json(file_json_path, save_path):
    file_json_path = Path(file_json_path)
    save_path = Path(save_path)

    with open(file_json_path, "r", encoding="utf-8") as f:
        file_json = json.load(f)

    def tag_name(node):
        return node.get("tag", "").split(":")[-1]

    def is_tag(node, name):
        return tag_name(node) == name

    def walk(node, tag=None):
        results = []
        if tag is None or is_tag(node, tag):
            results.append(node)
        for child in node.get("children", []):
            results.extend(walk(child, tag))
        return results

    def first_child(node, tag):
        for child in node.get("children", []):
            if is_tag(child, tag):
                return child
        return None

    def to_int(value, default=None):
        try:
            return int(value)
        except (TypeError, ValueError):
            return default

    def get_text_from_run(run):
        texts = []
        for child in run.get("children", []):
            if is_tag(child, "t"):
                texts.append(child.get("text", ""))
        return "".join(texts)

    def get_paragraph_text(p_node):
        texts = []
        for run in walk(p_node, "run"):
            text = get_text_from_run(run)
            if text:
                texts.append(text)
        return "".join(texts)

    header_root = file_json["header"]["data"]
    section_root = file_json["section"]["data"]
    settings_root = file_json["settings"]["data"]

    font_map = {}
    for font in walk(header_root, "font"):
        attrs = font.get("attrs", {})
        font_id = attrs.get("id")
        face = attrs.get("face")
        if font_id is not None and face:
            font_map[font_id] = face

    char_style_map = {}
    for char_pr in walk(header_root, "charPr"):
        attrs = char_pr.get("attrs", {})
        char_id = attrs.get("id")

        font_ref = first_child(char_pr, "fontRef")
        font_id = font_ref.get("attrs", {}).get("hangul") if font_ref else None

        italic_node = first_child(char_pr, "italic")
        bold_node = first_child(char_pr, "bold")
        underline_node = first_child(char_pr, "underline")
        strikeout_node = first_child(char_pr, "strikeout")

        height = to_int(attrs.get("height"))

        char_style_map[char_id] = {
            "id": char_id,
            "font_id": font_id,
            "font": font_map.get(font_id),
            "height": height,
            "size_px_guess": height // 100 if height is not None else None,
            "textColor": attrs.get("textColor"),
            "shadeColor": attrs.get("shadeColor"),
            "bold": bold_node is not None,
            "italic": italic_node is not None,
            "underline": underline_node.get("attrs", {}) if underline_node else None,
            "strikeout": strikeout_node.get("attrs", {}) if strikeout_node else None,
            "raw_attrs": attrs,
        }

    para_style_map = {}
    for para_pr in walk(header_root, "paraPr"):
        attrs = para_pr.get("attrs", {})
        para_id = attrs.get("id")

        align = None
        heading = None
        line_spacing = None

        for child in para_pr.get("children", []):
            cattrs = child.get("attrs", {})
            if is_tag(child, "align"):
                align = cattrs
            elif is_tag(child, "heading"):
                heading = cattrs
            elif is_tag(child, "lineSpacing"):
                line_spacing = cattrs

        para_style_map[para_id] = {
            "id": para_id,
            "align": align,
            "heading": heading,
            "lineSpacing": line_spacing,
            "raw_attrs": attrs,
        }

    def parse_text_runs(p_node):
        parsed_runs = []

        for run_node in p_node.get("children", []):
            if not is_tag(run_node, "run"):
                continue

            if walk(run_node, "tbl"):
                continue

            run_attrs = run_node.get("attrs", {})
            char_id = run_attrs.get("charPrIDRef")
            text = get_text_from_run(run_node)

            if text:
                parsed_runs.append({
                    "type": "text_run",
                    "text": text,
                    "charPrIDRef": char_id,
                    "style": char_style_map.get(char_id),
                    "raw_attrs": run_attrs,
                })

        return parsed_runs

    def child_attrs(node, tag):
        child = first_child(node, tag)
        return child.get("attrs", {}) if child else {}

    def parse_table(tbl_node):
        table = {
            "type": "table",
            "raw_attrs": tbl_node.get("attrs", {}),
            "raw_node": tbl_node,   # 중요: HWPX 재생성용 원본 표 XML 보존
            "rows": []
        }

        for tr_node in tbl_node.get("children", []):
            if not is_tag(tr_node, "tr"):
                continue

            row = {
                "type": "table_row",
                "raw_attrs": tr_node.get("attrs", {}),
                "cells": []
            }

            for tc_node in tr_node.get("children", []):
                if not is_tag(tc_node, "tc"):
                    continue

                cell = {
                    "type": "table_cell",
                    "raw_attrs": tc_node.get("attrs", {}),
                    "cellAddr": child_attrs(tc_node, "cellAddr"),
                    "cellSpan": child_attrs(tc_node, "cellSpan"),
                    "cellSz": child_attrs(tc_node, "cellSz"),
                    "paragraphs": []
                }

                for p_node in walk(tc_node, "p"):
                    p_attrs = p_node.get("attrs", {})
                    para_id = p_attrs.get("paraPrIDRef")

                    cell["paragraphs"].append({
                        "type": "paragraph",
                        "paraPrIDRef": para_id,
                        "styleIDRef": p_attrs.get("styleIDRef"),
                        "paragraph_style": para_style_map.get(para_id),
                        "runs": parse_text_runs(p_node),
                        "text": get_paragraph_text(p_node),
                        "raw_attrs": p_attrs,
                    })

                row["cells"].append(cell)

            table["rows"].append(row)

        return table

    paragraphs = []

    for p_index, p in enumerate(section_root.get("children", [])):
        if not is_tag(p, "p"):
            continue

        p_attrs = p.get("attrs", {})
        para_id = p_attrs.get("paraPrIDRef")
        runs = []
        layout = None

        tbl_nodes = walk(p, "tbl")

        for tbl_node in tbl_nodes:
            runs.append(parse_table(tbl_node))

        if not tbl_nodes:
            for child in p.get("children", []):
                if is_tag(child, "run"):
                    run_attrs = child.get("attrs", {})
                    char_id = run_attrs.get("charPrIDRef")
                    text = get_text_from_run(child)

                    if text:
                        runs.append({
                            "type": "text_run",
                            "text": text,
                            "charPrIDRef": char_id,
                            "style": char_style_map.get(char_id),
                            "raw_attrs": run_attrs,
                        })

        for child in p.get("children", []):
            if is_tag(child, "linesegarray"):
                linesegs = []

                for lineseg in child.get("children", []):
                    if is_tag(lineseg, "lineseg"):
                        a = lineseg.get("attrs", {})
                        linesegs.append({
                            "textpos": to_int(a.get("textpos")),
                            "vertpos": to_int(a.get("vertpos")),
                            "vertsize": to_int(a.get("vertsize")),
                            "textheight": to_int(a.get("textheight")),
                            "baseline": to_int(a.get("baseline")),
                            "spacing": to_int(a.get("spacing")),
                            "horzpos": to_int(a.get("horzpos")),
                            "horzsize": to_int(a.get("horzsize")),
                            "flags": to_int(a.get("flags")),
                            "raw_attrs": a,
                        })

                layout = {"linesegs": linesegs}

        paragraphs.append({
            "type": "paragraph",
            "index": p_index,
            "paraPrIDRef": para_id,
            "styleIDRef": p_attrs.get("styleIDRef"),
            "paragraph_style": para_style_map.get(para_id),
            "runs": runs,
            "layout": layout,
            "raw_attrs": p_attrs,
        })

    render_json = {
        "type": "hwpx_render_json",
        "source": {
            "section": file_json["section"].get("source_path"),
            "header": file_json["header"].get("source_path"),
            "settings": file_json["settings"].get("source_path"),
        },
        "maps": {
            "fonts": font_map,
            "char_styles": char_style_map,
            "para_styles": para_style_map,
        },
        "document": {
            "paragraphs": paragraphs,
            "settings_raw": settings_root,
        }
    }

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(render_json, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {save_path}")
    return render_json


from pathlib import Path
import json

def render_json_to_section_xml(render_json_path, save_path):
    """렌더링용 JSON 파일 경로 -> section0.xml 저장"""

    import xml.etree.ElementTree as ET

    render_json_path = Path(render_json_path)
    save_path = Path(save_path)

    with open(render_json_path, "r", encoding="utf-8") as f:
        render_json = json.load(f)

    def str_attrs(attrs):
        return {
            str(k): str(v)
            for k, v in attrs.items()
            if v is not None
        }

    def json_to_xml_elem(data):
        elem = ET.Element(data["tag"], str_attrs(data.get("attrs", {})))

        text = data.get("text", "")
        if text:
            elem.text = text

        for child in data.get("children", []):
            elem.append(json_to_xml_elem(child))

        return elem

    root = ET.Element("sec")

    paragraphs = render_json["document"]["paragraphs"]

    for paragraph in paragraphs:
        raw_p_attrs = paragraph.get("raw_attrs", {})

        p_attrs = {
            "id": raw_p_attrs.get("id", "0"),
            "paraPrIDRef": paragraph.get("paraPrIDRef", raw_p_attrs.get("paraPrIDRef", "0")),
            "styleIDRef": paragraph.get("styleIDRef", raw_p_attrs.get("styleIDRef", "0")),
            "pageBreak": raw_p_attrs.get("pageBreak", "0"),
            "columnBreak": raw_p_attrs.get("columnBreak", "0"),
            "merged": raw_p_attrs.get("merged", "0"),
        }

        p_elem = ET.SubElement(root, "p", str_attrs(p_attrs))

        for run in paragraph.get("runs", []):
            if run.get("type") == "table":
                run_elem = ET.SubElement(p_elem, "run", {"charPrIDRef": "0"})

                raw_node = run.get("raw_node")
                if raw_node:
                    run_elem.append(json_to_xml_elem(raw_node))

                t_elem = ET.SubElement(run_elem, "t")
                t_elem.text = ""

                continue

            run_attrs = run.get("raw_attrs", {})

            run_elem = ET.SubElement(
                p_elem,
                "run",
                str_attrs({
                    "charPrIDRef": run.get("charPrIDRef", run_attrs.get("charPrIDRef", "0"))
                })
            )

            text = run.get("text", "")

            if text:
                t_elem = ET.SubElement(run_elem, "t")
                t_elem.text = text

        layout = paragraph.get("layout") or {}
        linesegs = layout.get("linesegs", [])

        if linesegs:
            linesegarray_elem = ET.SubElement(p_elem, "linesegarray")

            for lineseg in linesegs:
                raw_attrs = lineseg.get("raw_attrs", {})

                lineseg_attrs = {
                    "textpos": lineseg.get("textpos", raw_attrs.get("textpos")),
                    "vertpos": lineseg.get("vertpos", raw_attrs.get("vertpos")),
                    "vertsize": lineseg.get("vertsize", raw_attrs.get("vertsize")),
                    "textheight": lineseg.get("textheight", raw_attrs.get("textheight")),
                    "baseline": lineseg.get("baseline", raw_attrs.get("baseline")),
                    "spacing": lineseg.get("spacing", raw_attrs.get("spacing")),
                    "horzpos": lineseg.get("horzpos", raw_attrs.get("horzpos")),
                    "horzsize": lineseg.get("horzsize", raw_attrs.get("horzsize")),
                    "flags": lineseg.get("flags", raw_attrs.get("flags")),
                }

                ET.SubElement(linesegarray_elem, "lineseg", str_attrs(lineseg_attrs))

    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ", level=0)

    save_path.parent.mkdir(parents=True, exist_ok=True)

    tree.write(save_path, encoding="utf-8", xml_declaration=True)

    print(f"저장 완료: {save_path}")
    return save_path

def render_json_to_html(render_json_path, save_path):
    """렌더링용 JSON 파일 -> HTML 파일"""

    render_json_path = Path(render_json_path)
    save_path = Path(save_path)

    with open(render_json_path, "r", encoding="utf-8") as f:
        render_json = json.load(f)

    def get_align(paragraph):
        para_style = paragraph.get("paragraph_style") or {}
        align = para_style.get("align") or {}
        value = align.get("horizontal") or align.get("textAlign") or align.get("align") or align.get("type")

        align_map = {
            "LEFT": "left",
            "CENTER": "center",
            "RIGHT": "right",
            "JUSTIFY": "justify",
            "BOTH": "justify",
        }

        return align_map.get(value, "left")

    def run_to_span(run):
        text = html.escape(run.get("text", ""))
        style = run.get("style") or {}

        css = []

        if style.get("font"):
            css.append(f"font-family: '{style['font']}'")

        if style.get("size_px_guess"):
            css.append(f"font-size: {style['size_px_guess']}px")

        if style.get("textColor"):
            css.append(f"color: {style['textColor']}")

        if style.get("bold"):
            css.append("font-weight: bold")

        if style.get("italic"):
            css.append("font-style: italic")

        return f'<span style="{"; ".join(css)}">{text}</span>'

    def hwpunit_to_px(value):
        try:
            return max(20, int(value) / 100)
        except (TypeError, ValueError):
            return None

    def table_to_html(table):
        rows_html = []

        for row in table.get("rows", []):
            cells_html = []

            for cell in row.get("cells", []):
                span = cell.get("cellSpan") or {}
                size = cell.get("cellSz") or {}

                colspan = span.get("colSpan", "1")
                rowspan = span.get("rowSpan", "1")

                width_px = hwpunit_to_px(size.get("width"))

                td_style = [
                    "border:1px solid #333",
                    "padding:6px",
                    "vertical-align:top"
                ]

                if width_px:
                    td_style.append(f"width:{width_px}px")

                para_html = []

                for p in cell.get("paragraphs", []):
                    runs = p.get("runs", [])
                    text = p.get("text", "")

                    if runs:
                        content = "".join(run_to_span(r) for r in runs)
                    else:
                        content = html.escape(text) if text else "&nbsp;"

                    para_html.append(f'<p style="margin:0 0 4px 0;">{content}</p>')

                cell_content = "".join(para_html) if para_html else "&nbsp;"

                cells_html.append(
                    f'<td colspan="{colspan}" rowspan="{rowspan}" style="{"; ".join(td_style)}">{cell_content}</td>'
                )

            rows_html.append(f"<tr>{''.join(cells_html)}</tr>")

        return f"""
<table style="border-collapse:collapse; margin:8px 0; table-layout:fixed;">
{''.join(rows_html)}
</table>
"""

    body_parts = []

    for paragraph in render_json["document"]["paragraphs"]:
        runs = paragraph.get("runs", [])

        if not runs:
            body_parts.append("<p>&nbsp;</p>")
            continue

        align = get_align(paragraph)
        inline_parts = []

        for run in runs:
            if run.get("type") == "text_run":
                inline_parts.append(run_to_span(run))
            elif run.get("type") == "table":
                body_parts.append(table_to_html(run))

        if inline_parts:
            body_parts.append(
                f'<p style="text-align:{align}; margin:0 0 8px 0;">{"".join(inline_parts)}</p>'
            )

    html_text = f"""<!doctype html>
        <html lang="ko">
        <head>
        <meta charset="utf-8">
        <title>HWPX Render</title>
        </head>
        <body>
        {chr(10).join(body_parts)}
        </body>
        </html>
"""

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        f.write(html_text)

    print(f"HTML 저장 완료: {save_path}")
    return save_path


In [50]:
from pathlib import Path

# 1. 템플릿 불러오기
input_file_path = Path(r"HWPX 파일\ex_대목차+본문.hwpx")
output_root = Path("step4_output")
xml_dir = output_root / "xml"

output_root.mkdir(parents=True, exist_ok=True)

print("현재 경로:", Path.cwd())
print("입력 파일:", input_file_path)

# 2. 템플릿 HWPX 압축 해제
extract_hwpx(input_file_path, xml_dir)

# 3. 템플릿 json으로 변환
file_json_path = output_root / f"{input_file_path.stem}.json"

file_json = hwpx_xml_to_json(
    xml_dir=xml_dir,
    save_path=file_json_path
)

# 4. 템플릿 렌더링 json으로 변환
render_json_path = output_root / f"{input_file_path.stem}_render.json"

render_json = make_render_json(
    file_json_path=file_json_path,
    save_path=render_json_path
)

print(render_json["document"]["paragraphs"][0])

# 5.1 html로 렌더링 해보기
html_path = output_root / f"{input_file_path.stem}.html"

render_json_to_html(
    render_json_path=render_json_path,
    save_path=html_path
)

# 5.2 렌더링 json 변경해보기
# 필요하면 여기에서 render_json 파일을 수정한 뒤 다시 저장
# 예:
render_json["document"]["paragraphs"][0]["text"] = "변경할 내용"

# 6. 렌더링 json -> 한글 section0.xml로 변환
section_xml_path = xml_dir / "Contents" / "section0.xml"

render_json_to_section_xml(
    render_json_path=render_json_path,
    save_path=section_xml_path
)

# 7. XML -> HWPX 압축
template_dir = Path("빈파일_xml")
output_hwpx = output_root / "result.hwpx"

xml_to_hwpx(
    xml_dir=xml_dir,
    template_dir=template_dir,
    output_hwpx=output_hwpx
)

print("완료:", output_hwpx)

현재 경로: c:\Users\hyun\Documents\Project\0_PROJECT\micro-labs\week08a_hwpx
입력 파일: HWPX 파일\ex_대목차+본문.hwpx
저장 완료: step4_output\ex_대목차+본문.json
저장 완료: step4_output\ex_대목차+본문_render.json
{'type': 'paragraph', 'index': 0, 'paraPrIDRef': '0', 'styleIDRef': '0', 'paragraph_style': {'id': '0', 'align': {'horizontal': 'JUSTIFY', 'vertical': 'BASELINE'}, 'heading': {'type': 'NONE', 'idRef': '0', 'level': '0'}, 'lineSpacing': None, 'raw_attrs': {'id': '0', 'tabPrIDRef': '0', 'condense': '0', 'fontLineHeight': '0', 'snapToGrid': '1', 'suppressLineNumbers': '0', 'checked': '0', 'textDir': 'LTR'}}, 'runs': [{'type': 'table', 'raw_attrs': {'id': '1101173222', 'zOrder': '0', 'numberingType': 'TABLE', 'textWrap': 'TOP_AND_BOTTOM', 'textFlow': 'BOTH_SIDES', 'lock': '0', 'dropcapstyle': 'None', 'pageBreak': 'CELL', 'repeatHeader': '1', 'rowCnt': '9', 'colCnt': '2', 'cellSpacing': '0', 'borderFillIDRef': '3', 'noAdjust': '0'}, 'raw_node': {'tag': 'tbl', 'attrs': {'id': '1101173222', 'zOrder': '0', 'numbering